# Module 2: The RAG Analyst (ChromaDB + MCP) 🔍

Welcome to Module 2! In this session, we will build a **Retrieval Augmented Generation (RAG)** agent using modern tools:

1.  **ChromaDB**: A popular open-source Vector Database.
2.  **Model Context Protocol (MCP)**: A standard way for AI models to connect to external data and tools.

### Why MCP?
Instead of writing custom Python functions to wrapp every API (like we did in Module 1), MCP lets us use standard connectors. We will use the `chroma-mcp` connector to let our agent talk to the database without writing query logic ourselves!

### Learning Objectives
-   Ingest text documents into a Vector Database.
-   Understand how MCP simplifies tool integration.
-   Build an agent that can answer questions from private data.

## 1. Setup and Installation

We need:`google-adk[mcp]`, `chromadb`, and `uv` (a fast Python package manager).
We also install `python-dotenv` to load our API key securely.

In [ ]:
%pip install "google-adk[mcp]" chromadb uv python-dotenv

In [ ]:
import os
import logging
from dotenv import load_dotenv

logging.basicConfig(level=logging.ERROR)

# Load API key from .env.local in the project root
load_dotenv('../.env.local')

if "GOOGLE_API_KEY" not in os.environ:
    print("⚠️ Warning: GOOGLE_API_KEY not found. Please check your .env.local file.")
else:
    print("✅ API Key loaded.")

## 2. The Knowledge Base

We have 3 text files in this folder representing our private knowledge:
-   `investment_policy.txt`
-   `market_outlook_2025.txt`
-   `lumenridge_tech_strategy.txt`

Let's preview one.

In [ ]:
with open("lumenridge_tech_strategy.txt", "r", encoding="utf-8") as f:
    print(f.read())

## 3. Ingestion: Creating the Vector Database

We need to turn these text files into **Vectors** (numbers) and store them in **ChromaDB** so the agent can search them.

We've prepared a script `ingest.py` that:
1.  Reads all `.txt` files.
2.  Creates a local database in `./rag_agent/demo_rag_db` (inside our agent folder).
3.  Rebuilds the `demo_docs` collection from those files, removing stale documents from earlier runs.
4.  Embeds and saves the current text. The step is safe to rerun.

Let's run it:

In [ ]:
# Create the agent directory first
!mkdir -p rag_agent

# Run ingestion
!python ingest.py

## 4. Connecting via MCP

Now for the magic. We don't need to write a `search()` function. We simply tell the Agent:

> "Here is an MCP server running Chroma. Use it."

We use `McpToolset` to make this connection. We use `uvx` (from the `uv` package) to run the `chroma-mcp` server instantly without messing with our local pip environment.

In [ ]:
from google.adk.agents import Agent
from google.adk.tools import McpToolset
from google.adk.tools.mcp_tool.mcp_session_manager import StdioConnectionParams, StdioServerParameters
import os

# Calculate absolute path so this works from any execution context
# When running directly in notebook, __file__ might not exist, but we assume relative path
# For the saved file later, we will use absolute path.
db_path = "./rag_agent/demo_rag_db" 
if not os.path.exists(db_path):
    # Fallback if running from root relative
    db_path = "./rag_agent/demo_rag_db"

# Define the Tool
chroma_tool = McpToolset(
    connection_params=StdioConnectionParams(
        server_params=StdioServerParameters(
            command="uvx", 
            args=[
                "chroma-mcp", 
                "--client-type", "persistent",
                "--data-dir", db_path, 
            ],
        )
    )
)

# Create the Agent
# We name it 'root_agent' so it can be picked up by 'adk web'
root_agent = Agent(
    model='gemini-3.5-flash-lite',
    name='rag_agent',
    instruction="""You are a RAG Analyst. 
    Use the `chroma_query_documents` (or similar) tool to answer questions based on the user's document collection.
    The collection name is 'demo_docs'.
    If you find relevant info, summarize it concisely.
    """,
    tools=[chroma_tool]
)

print("✅ Agent defined (in memory)!")

## 5. Running the Agent (Interactive)

Let's chat! Try asking questions about the documents we just ingested.

In [ ]:
from google.adk.runners import Runner
from google.adk.sessions.in_memory_session_service import InMemorySessionService
from google.genai import types
import uuid

runner = Runner(
    agent=root_agent,
    app_name="rag_mcp_app",
    session_service=InMemorySessionService(),
    auto_create_session=True
)

async def run_chat():
    user_id = "user_1"
    session_id = str(uuid.uuid4())
    print(f"starting session: {session_id}")

    while True:
        text = input("User (type 'quit' to exit): ")
        if text.lower() in ["quit", "exit"]:
            break
        
        print("   (Thinking...)")
        async for event in runner.run_async(
            user_id=user_id,
            session_id=session_id,
            new_message=types.Content(role="user", parts=[types.Part(text=text)])
        ):
            if event.content and event.content.parts:
                for part in event.content.parts:
                    if part.text:
                        print(f"Agent: {part.text}")


# Uncomment to run in notebook
# await run_chat()

## 6. Going Production: ADK Web

To run this as a real app, we need to save the agent code to a file: `rag_agent/agent.py`.
We also need to make sure the folder is structured so `adk` can find it.

In [ ]:
%%writefile rag_agent/agent.py
import os
from dotenv import load_dotenv

# Load .env.local from project root (2 dirs up from this file)
current_dir = os.path.dirname(os.path.abspath(__file__))
project_root = os.path.abspath(os.path.join(current_dir, "../../"))
env_path = os.path.join(project_root, ".env.local")
load_dotenv(env_path)

if "GOOGLE_API_KEY" not in os.environ:
    print(f"⚠️ Warning: GOOGLE_API_KEY not found in {env_path}")

from google.adk.agents import Agent
from google.adk.tools import McpToolset
from google.adk.tools.mcp_tool.mcp_session_manager import StdioConnectionParams, StdioServerParameters

# Calculate absolute path to the database to ensure it works from any CWD
# The DB is in the same folder as this agent file
db_dir = os.path.join(os.path.dirname(os.path.abspath(__file__)), "demo_rag_db")

# Tool Definition
chroma_tool = McpToolset(
    connection_params=StdioConnectionParams(
        server_params=StdioServerParameters(
            command="uvx", 
            args=[
                "chroma-mcp", 
                "--client-type", "persistent",
                "--data-dir", db_dir, # Use absolute path
            ],
        )
    )
)

# Agent Definition (root_agent)
root_agent = Agent(
    model='gemini-3.5-flash-lite',
    name='rag_agent',
    instruction="""You are a RAG Analyst. 
    Use the `chroma_query_documents` (or similar) tool to answer questions based on the user's document collection.
    The collection name is 'demo_docs'.
    If you find relevant info, summarize it concisely.
    """,
    tools=[chroma_tool]
)

Now you can run the web UI!

```bash
# Run from the terminal
adk web module_02
```

This works because `module_02` now contains a subfolder `rag_agent` which has our `agent.py`.